[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Relationships &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: `Team` and `Hero` with the relationships that hold each
other, and the eight heroes and three teams loaded. Run it first. The tasks follow one another, and
the last cell removes the scratch folder.


In [1]:
import contextlib
import logging
import re
import shutil
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, func, insert
from sqlalchemy.orm.exc import DetachedInstanceError
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


@contextlib.contextmanager
def catching():
    """Collects warnings instead of printing them: Python prints one with the file that raised it."""
    with warnings.catch_warnings(record=True) as raised:
        warnings.simplefilter("always")
        yield raised


class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again

def run_python(path):
    """Run a file in a Python of its own and print what it printed."""
    done = subprocess.run([sys.executable, path], capture_output=True, text=True)
    print(done.stdout.strip() or done.stderr.strip().splitlines()[-1])


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes in",
          len(session.exec(select(Team)).all()), "teams")


sqlmodel 0.0.42 | 8 heroes in 3 teams


**1.** Every team, and who is on it.


In [2]:
with Session(engine) as session:
    for team in session.exec(select(Team).order_by(Team.name)):
        print(f"  {team.name:<16}", sorted(hero.name for hero in team.heroes) or "nobody")


  Preventers       ['Captain North America', 'Rusty-Man', 'Spider-Boy', 'Tarantula']
  Wakaland Guard   nobody
  Z-Force          ['Black Lion', 'Deadpond', 'Dr. Weird']


One `select` for the teams, and the heroes came from the relationship. Wakaland Guard has nobody, so
its list is empty.


**2.** A team and two heroes, added as one.


In [3]:
with Session(engine) as session:
    patrol = Team(name="Sky Patrol", headquarters="Cloud Base")
    patrol.heroes.append(Hero(name="Cloud Runner", secret_name="Ines Vela", age=28))
    patrol.heroes.append(Hero(name="Storm Kite", secret_name="Milo Vance", age=33))
    session.add(patrol)                                             # the heroes come with it
    session.commit()
    session.refresh(patrol)
    print("team id:", patrol.id)
    for hero in sorted(patrol.heroes, key=lambda hero: hero.name):
        print(f"  {hero.name:<14} id {hero.id} | team_id {hero.team_id}")


team id: 4
  Cloud Runner   id 9 | team_id 4
  Storm Kite     id 10 | team_id 4


Only the team was added. The heroes were written with it and their `team_id` was filled in from the
id the database gave the team.


**3.** A hero moved, with both lists watching.


In [4]:
with Session(engine) as session:
    patrol = session.exec(select(Team).where(Team.name == "Sky Patrol")).one()
    preventers = session.exec(select(Team).where(Team.name == "Preventers")).one()
    runner = session.exec(select(Hero).where(Hero.name == "Cloud Runner")).one()

    print("before: patrol", sorted(h.name for h in patrol.heroes))
    print("        preventers", len(preventers.heroes), "heroes")
    runner.team = preventers
    print("after : patrol", sorted(h.name for h in patrol.heroes))
    print("        preventers", len(preventers.heroes), "heroes")
    session.commit()


before: patrol ['Cloud Runner', 'Storm Kite']
        preventers 4 heroes
after : patrol ['Storm Kite']
        preventers 5 heroes


One assignment, and both lists changed before anything was committed: that is `back_populates`.


**4.** What reading a relationship sends.


In [5]:
engine.echo = True
with Session(engine) as session:
    team = session.get(Team, 1)
    heroes = [hero.name for hero in team.heroes]
engine.echo = False
# Two: one SELECT for the team, and one for its heroes when team.heroes was read.


    BEGIN (implicit)
    SELECT team.id AS team_id, team.name AS team_name, team.headquarters AS team_headquarters
    FROM team
    WHERE team.id = ?
    values: (1,)
    SELECT hero.id AS hero_id, hero.name AS hero_name, hero.secret_name AS hero_secret_name, hero.age AS hero_age, hero.team_id AS hero_team_id
    FROM hero
    WHERE ? = hero.team_id
    values: (1,)
    ROLLBACK


The second statement is sent at the moment the list is read, and not when the team was loaded.


**5.** The same count, two ways.


In [6]:
with Session(engine) as session:
    by_query = {name: heroes for name, heroes in
                session.exec(select(Team.name, func.count(Hero.id)).join(Hero, isouter=True).group_by(Team.name))}
    by_attribute = {team.name: len(team.heroes) for team in session.exec(select(Team))}

print("by query    :", dict(sorted(by_query.items())))
print("by attribute:", dict(sorted(by_attribute.items())))
print("they agree  :", by_query == by_attribute)


by query    : {'Preventers': 5, 'Sky Patrol': 1, 'Wakaland Guard': 0, 'Z-Force': 3}
by attribute: {'Preventers': 5, 'Sky Patrol': 1, 'Wakaland Guard': 0, 'Z-Force': 3}
they agree  : True


The query asked the database once; the attribute sent one query for every team. They agree, and the
**Loading and N+1** notebook is about the difference in what they cost.


**6.** Sky Patrol disbanded.


In [7]:
with Session(engine) as session:
    patrol = session.exec(select(Team).where(Team.name == "Sky Patrol")).one()
    theirs = sorted(hero.name for hero in patrol.heroes)
    session.delete(patrol)
    session.commit()

with Session(engine) as session:
    kite = session.exec(select(Hero).where(Hero.name == "Storm Kite")).one()
    print("on the team when it went:", theirs)
    print("Storm Kite is still here, with team_id", kite.team_id)
    print("heroes in the table     :", len(session.exec(select(Hero)).all()))


on the team when it went: ['Storm Kite']
Storm Kite is still here, with team_id None
heroes in the table     : 10


Nothing was deleted but the team. `Team` says nothing about what happens to its heroes, so the
session set their `team_id` to null, and Storm Kite is a hero on no team. Cloud Runner was moved to
the Preventers in task 3 and kept that team, which is why only one hero was left without one.

Last, the engine lets go of the file, and this cell removes the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Relationships](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/08-relationships.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
